In [1]:
import pandas as pd

df = pd.read_csv("../../data/processed/cleaned_data.csv")

df = df[["processed_text", "target"]]
df.head()

,processed_text,target
0,deeds reason earthquake may allah forgive us,1
1,forest fire near la ronge sask canada,1
2,residents asked shelter place notified officer...,1
3,13000 people receive wildfires evacuation orde...,1
4,got sent photo ruby alaska smoke wildfires pou...,1


## Pre-Trained Embedding Model: wiki-words

In [2]:
import tensorflow as tf
import numpy as np
import kagglehub

model_path = kagglehub.model_download("google/wiki-words/tensorFlow2/250")
embed = tf.saved_model.load(model_path)


/home/tresor/.conda/envs/nlp_disaster_tweets/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0000 00:00:1790266643.593754  140564 cpu_allocator_impl.cc:82] Allocation of 1009375000 exceeds 10% of free system memory.


### Preprocessing Sequences and Zero-Padding

In [3]:
# embedding helper function
def get_word2vec_enc(texts):
    encoded_texts = []
    for text in texts:
        tokens = text.split(" ")
        word2vec_embedding = embed(tokens)
        encoded_texts.append(word2vec_embedding)
    
    return encoded_texts

# text with the max length in df
def get_max_len(df):
    max_len = 0
    for row in df["processed_text"]:
        tokens = row.split(" ")
        if len(tokens) > max_len:
            max_len = len(tokens)
    return max_len


# zero-padding
def get_padded_encoded_texts(encoded_texts, max_len):
    padded_texts_encoding = []
    for enc_text in encoded_texts:
        zero_padding_count = max_len - enc_text.shape[0]
        pad = np.zeros((1, 250))
        for i in range(zero_padding_count):
            enc_text = np.concatenate((pad, enc_text), axis=0)
        padded_texts_encoding.append(enc_text)
    return padded_texts_encoding

# preprocessing
def preprocess(df, max_len):
    
    texts = df["processed_text"].tolist()
    encoded_texts = get_word2vec_enc(texts)
    
    padded_encoded_texts = get_padded_encoded_texts(encoded_texts, max_len)
    
    sentiments = df["target"].tolist()
    
    x = np.array(padded_encoded_texts)
    y = np.array(sentiments)
    
    return x, y
    

max_length = get_max_len(df)

print("Max length of the dataset is:", max_length)

x, y = preprocess(df, max_length)
print("x:", x.shape, "; y:", y.shape)

Max length of the dataset is: 25
x: (7613, 25, 250) ; y: (7613,)


In [4]:
import random

pos_indices = df[df['target'] == 1].index.tolist()
random_pos_idx = random.choice(pos_indices)

print("--- Random Positive Tweet ---")
print(f"Text: {df['processed_text'].iloc[random_pos_idx]}")
print(f"\nEmbedding Matrix Shape: {x[random_pos_idx].shape}")

print("\nFull Embedding Matrix (Padded):")
print(x[random_pos_idx])

--- Random Positive Tweet ---
Text: _gaabyx got purple activist thought drought

Embedding Matrix Shape: (25, 250)

Full Embedding Matrix (Padded):
[[ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 ...
 [-0.04335316 -0.07457704 -0.07139074 ...  0.07947811 -0.09739878
  -0.1044645 ]
 [-0.02612394 -0.05889868  0.07008632 ... -0.01321644 -0.01440209
  -0.03093668]
 [-0.04181252 -0.1166335   0.0708295  ...  0.01064647 -0.01117454
   0.08920804]]


In [5]:
import numpy as np


np.savez_compressed('dataset_preprocessed.npz', X_data=x, Y_data=y)